# 03 — MinIO (Iceberg data files) → Azure ADLS

**RF-03**: move the Parquet backing the Iceberg table from MinIO to the group's
folder in Azure Data Lake Storage, using **dlt**.

The source prefix is not hardcoded: it is asked of the Nessie catalog.

The verification cell uses **adlfs** (the Azure filesystem library) with the
same credentials dlt used — so the evidence does not depend on the Azure portal.

In [3]:
from __future__ import annotations

import sys
from urllib.parse import urlparse

sys.path.insert(0, "/home/jovyan/scripts")

import dlt
from dlt.sources.filesystem import readers

import config as cfg

PIPELINE_NAME = "minio_to_azure"


In [4]:
def secret(path: str, default=None):
    """Read a value from .dlt/secrets.toml without duplicating it in code."""
    try:
        return dlt.secrets[path]
    except Exception:
        return default


def iceberg_data_prefix() -> str:
    """Ask Nessie where the Iceberg table stores its data files."""
    catalog = cfg.iceberg_catalog()
    assert catalog.table_exists(cfg.TABLE_IDENTIFIER), (
        f"Table {cfg.TABLE_IDENTIFIER} does not exist in Nessie. "
        "Run notebook 02 first."
    )
    location = catalog.load_table(cfg.TABLE_IDENTIFIER).location()
    return f"{location.rstrip('/')}/data/"


In [5]:
bucket_url = secret(f"{PIPELINE_NAME}.destination.filesystem.bucket_url")
assert bucket_url, (
    f"FAIL: bucket_url missing from dlt/secrets.toml under "
    f"[{PIPELINE_NAME}.destination.filesystem]"
)

source_prefix = iceberg_data_prefix()
print(f"Source     : {source_prefix}")
print(f"Destination: {bucket_url}")


Source     : s3://nyc-taxi-iceberg/nyc_taxi/yellow_tripdata_2025_01_89f30148-8855-4aa6-be06-8061670c3b93/data/
Destination: abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_2


Source     : s3://nyc-taxi-iceberg/nyc_taxi/yellow_tripdata_2025_01_4ba6d1f7-dd1d-45d0-9e42-830d1b373f5b/data/
Destination: abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_2


In [6]:
reader = readers(bucket_url=source_prefix, file_glob="**/*.parquet").read_parquet()
reader = reader.with_name(cfg.ICEBERG_TABLE)

pipeline = dlt.pipeline(
    pipeline_name=PIPELINE_NAME,
    destination="filesystem",
    dataset_name=cfg.NESSIE_NAMESPACE,
)

load_info = pipeline.run(
    reader,
    loader_file_format="parquet",
    write_disposition="replace",
)
print(load_info)
print(pipeline.last_trace.last_normalize_info)


2026-09-03 02:24:26,050|[INFO]|4643|138500491134400|dlt|pipeline.py|_restore_state_from_destination:1947|The state was restored from the destination filesystem (dlt.destinations.filesystem):nyc_taxi
2026-09-03 02:25:35,003|[INFO]|4643|138500491134400|dlt|pool_runner.py|create_pool:203|Created none pool with 1 workers
2026-09-03 02:25:35,004|[INFO]|4643|138500491134400|dlt|normalize.py|run:309|Running file normalizing
2026-09-03 02:25:35,005|[INFO]|4643|138500491134400|dlt|normalize.py|run:312|Found 1 load packages
2026-09-03 02:25:35,008|[INFO]|4643|138500491134400|dlt|normalize.py|run:335|Found 1 files in schema minio_to_azure load_id 1788402266.0825484
2026-09-03 02:25:35,011|[INFO]|4643|138500491134400|dlt|normalize.py|spool_schema_files:298|Created new load package 1788402266.0825484 on loading volume with 1 files
2026-09-03 02:25:35,016|[INFO]|4643|138500491134400|dlt|worker.py|_get_items_normalizer:137|A file format for table yellow_tripdata_2025_01 was specified to parquet in th

Pipeline minio_to_azure load step finished in 12 minutes and 11.62 seconds
1 load package(s) were loaded to destination filesystem and into dataset nyc_taxi
The filesystem destination used abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_2 location to store data
Load package 1788402266.0825484 is LOADED and contains no failed jobs
Normalized data for the following tables:
- yellow_tripdata_2025_01: 3475226 row(s)

Load package 1788402266.0825484 is NORMALIZED and NOT YET LOADED to the destination and contains no failed jobs


2026-09-03 01:50:01,014|[INFO]|152|131262377461184|dlt|normalize.py|run:309|Running file normalizing


2026-09-03 01:50:01,015|[INFO]|152|131262377461184|dlt|normalize.py|run:312|Found 1 load packages


2026-09-03 01:50:01,018|[INFO]|152|131262377461184|dlt|normalize.py|run:335|Found 1 files in schema minio_to_azure load_id 1788400136.2320294


...

2026-09-03 02:03:58,554|[INFO]|152|131262377461184|dlt|load.py|start_new_jobs:327|Will load additional 0, creating jobs


2026-09-03 02:03:59,137|[INFO]|152|131262377461184|dlt|load.py|complete_package:657|All jobs completed, archiving package 1788400136.2320294 with aborted set to False


2026-09-03 02:03:59,138|[INFO]|152|131262377461184|dlt|pool_runner.py|run_pool:279|Closing processing pool


2026-09-03 02:03:59,139|[INFO]|152|131262377461184|dlt|pool_runner.py|run_pool:282|Processing pool closed


Pipeline minio_to_azure load step finished in 11 minutes and 51.71 seconds
1 load package(s) were loaded to destination filesystem and into dataset nyc_taxi
The filesystem destination used abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_2 location to store data
Load package 1788400136.2320294 is LOADED and contains no failed jobs
Normalized data for the following tables:
- yellow_tripdata_2025_01: 3475226 row(s)

Load package 1788400136.2320294 is NORMALIZED and NOT YET LOADED to the destination and contains no failed jobs


In [5]:
import adlfs

cfg.banner("Verification | files in Azure ADLS (via adlfs, no portal needed)")

account_name = secret(
    f"{PIPELINE_NAME}.destination.filesystem.credentials.azure_storage_account_name"
)
account_key = secret(
    f"{PIPELINE_NAME}.destination.filesystem.credentials.azure_storage_account_key"
)

parsed = urlparse(bucket_url)          # abfss://container@account.dfs.../path
container = parsed.netloc.split("@", 1)[0]
path = parsed.path.lstrip("/")

filesystem = adlfs.AzureBlobFileSystem(
    account_name=account_name,
    account_key=account_key,
)

entries = filesystem.find(f"{container}/{path}")
assert entries, f"FAIL: nothing found under {bucket_url}"

total = 0
for entry in entries:
    size = filesystem.info(entry).get("size", 0)
    total += size
    print(f"  {entry}  ({size / 1e6:,.2f} MB)")
print(f"\nFiles in Azure: {len(entries)}  |  total {total / 1e6:,.1f} MB")



  Verification | files in Azure ADLS (via adlfs, no portal needed)


  clase-4-dlt/GRUPO_2/nyc_taxi/_dlt_loads/minio_to_azure__1788398277.065915.jsonl  (0.00 MB)


  clase-4-dlt/GRUPO_2/nyc_taxi/_dlt_loads/minio_to_azure__1788400136.2320294.jsonl  (0.00 MB)


  clase-4-dlt/GRUPO_2/nyc_taxi/_dlt_pipeline_state/minio_to_azure__1788398277.065915__1bf8bc25022766b3e1677c490bf1d998480bf0a6c02875dea724787aad326163.jsonl  (0.00 MB)


  clase-4-dlt/GRUPO_2/nyc_taxi/_dlt_version/minio_to_azure__1788398469.2516515__1bf8bc25022766b3e1677c490bf1d998480bf0a6c02875dea724787aad326163.jsonl  (0.01 MB)


  clase-4-dlt/GRUPO_2/nyc_taxi/init  (0.00 MB)


  clase-4-dlt/GRUPO_2/nyc_taxi/yellow_tripdata_2025_01/1788400136.2320294.c3b5060a26.parquet  (145.58 MB)

Files in Azure: 6  |  total 145.6 MB
